# Week 15: Graph Neural Networks & Geometric Deep Learning
## Notebook 01: Graph Fundamentals, Adjacency, and the Graph Laplacian from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In traditional Deep Learning (CNNs, RNNs), data lives on regular Euclidean grids:
- Images are regular 2D pixel grids with uniform spacing.
- Text and audio are 1D sequential grids.

However, real-world data is non-Euclidean:
- **Social Networks:** Users have arbitrary numbers of connections (irregular degree).
- **Molecules & Drugs:** Atoms have varying chemical bonds and 3D spatial geometries.
- **Financial Transactions:** Fraud rings form arbitrary cyclic and bipartite transaction graphs.

To operate on graphs mathematically, we must represent topological connectivity with matrices. The core foundation of Spectral Graph Theory is the **Graph Laplacian**.

---

### 2. Mathematical Foundations
A graph $G = (V, E)$ consists of $N = |V|$ nodes and $|E|$ edges.

1. **Adjacency Matrix ($A \in \mathbb{R}^{N \times N}$):**
   $$A_{ij} = \begin{cases} 1 & \text{if } (i, j) \in E \\ 0 & \text{otherwise} \end{cases}$$
   For undirected graphs, $A = A^T$.

2. **Degree Matrix ($D \in \mathbb{R}^{N \times N}$):**
   Diagonal matrix where $D_{ii} = \sum_{j=1}^N A_{ij} = \text{deg}(v_i)$.

3. **Unnormalized Graph Laplacian ($L \in \mathbb{R}^{N \times N}$):**
   $$L = D - A$$
   For any signal vector $x \in \mathbb{R}^N$ defined on the graph nodes:
   $$x^T L x = \frac{1}{2} \sum_{i,j=1}^N A_{ij} (x_i - x_j)^2$$
   This proves $L$ is **Positive Semi-Definite (PSD)**, with eigenvalues $\lambda_1 = 0 \le \lambda_2 \le \dots \le \lambda_N$.
   - $\lambda_1 = 0$ corresponds to the constant eigenvector.
   - The multiplicity of eigenvalue 0 equals the number of connected components in the graph!
   - $\lambda_2$ is the **Fiedler value** (algebraic connectivity), measuring how easily the graph can be partitioned.

4. **Symmetric Normalized Laplacian ($L_{sym}$):**
   $$L_{sym} = D^{-1/2} L D^{-1/2} = I_N - D^{-1/2} A D^{-1/2}$$
   where $(D^{-1/2})_{ii} = 1 / \sqrt{D_{ii}}$. All eigenvalues lie in $[0, 2]$.

In [ ]:
import numpy as np

# Step 1: Define a 6-node social / citation network
# Graph structure:
# Node 0 - Node 1 - Node 2 (Community A)
#           |       |
#         Node 3 - Node 4 - Node 5 (Community B)

N = 6
edges = [
    (0, 1), (1, 2), (0, 2),
    (1, 3), (2, 4),
    (3, 4), (4, 5), (3, 5)
]

# Step 2: Build Adjacency Matrix A
A = np.zeros((N, N), dtype=float)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0  # Undirected

print("Adjacency Matrix A (6x6):")
print(A)

# Step 3: Compute Degree Matrix D
degrees = np.sum(A, axis=1)
D = np.diag(degrees)
print("\nDegree Matrix D:")
print(D)

# Step 4: Unnormalized Laplacian L = D - A
L = D - A
print("\nUnnormalized Laplacian L = D - A:")
print(L)

# Step 5: Spectral Decomposition of Laplacian
eigenvals, eigenvecs = np.linalg.eigh(L)
print("\nEigenvalues of L (Sorted):")
for i, val in enumerate(eigenvals):
    print(f"  lambda_{i}: {val:.4f}")

fiedler_val = eigenvals[1]
fiedler_vector = eigenvecs[:, 1]
print(f"\nFiedler Value (Algebraic Connectivity): {fiedler_val:.4f}")
print("Fiedler Vector (Partitioning indicator):", np.round(fiedler_vector, 3))

# Partition based on Fiedler vector sign
cluster_A = np.where(fiedler_vector < 0)[0]
cluster_B = np.where(fiedler_vector >= 0)[0]
print(f"Spectral Partition -> Cluster A: {cluster_A}, Cluster B: {cluster_B}")

# Step 6: Symmetric Normalized Laplacian L_sym = I - D^(-1/2) A D^(-1/2)
D_inv_sqrt = np.diag(1.0 / np.sqrt(degrees))
L_sym = np.eye(N) - (D_inv_sqrt @ A @ D_inv_sqrt)
print("\nSymmetric Normalized Laplacian L_sym:")
print(np.round(L_sym, 3))

### 3. Step-by-Step Code Walkthrough
1. **Adjacency Construction:** We initialized a $6 \times 6$ zero matrix and filled symmetric entries for each edge $(u, v)$.
2. **Degree Matrix Calculation:** Summing rows with `np.sum(A, axis=1)` yields the degree of each node, diagonalized with `np.diag`.
3. **Laplacian Matrix Formulation:** Subtracting $A$ from $D$ yields $L$. Notice diagonal entries are degrees, while off-diagonals are $-1$ or $0$.
4. **Eigen-Decomposition:** Using `np.linalg.eigh(L)` (for symmetric matrices) yields non-negative eigenvalues.
   - $\lambda_0 \approx 0.0000$ confirms the graph is fully connected.
   - $\lambda_1$ (Fiedler value) measures connectivity bottleneck. The signs of the Fiedler vector cleanly bisect the graph into its two natural communities: ${0, 1, 2}$ and ${3, 4, 5}$.
5. **Normalized Laplacian:** Scaling by $D^{-1/2}$ normalizes for high-degree hub nodes, preventing nodes with hundreds of edges from exploding gradients in neural operations.

---

### 4. Industry Applications & Trade-offs
- **Spectral Clustering:** Unsupervised customer segmentation and fraud clique discovery.
- **Image Segmentation:** Treating image pixels as graph nodes with similarity edges; normalized cuts partition foreground from background.
- **Trade-off:** Computing full eigen-decomposition scales at $O(N^3)$, which is intractable for graphs with millions of nodes. This directly motivated **spatial message-passing GNNs** like GCN and GraphSAGE.

---

### 5. Top Interview Questions & Answers
**Q1: Why is the smallest eigenvalue of an unnormalized Graph Laplacian always zero?**
> **Answer:** Let $\mathbf{1} = [1, 1, \dots, 1]^T$ be the all-ones vector. The $i$-th row of $L \mathbf{1}$ is $D_{ii}(1) - \sum_j A_{ij}(1) = \text{deg}(v_i) - \text{deg}(v_i) = 0$. Thus, $L \mathbf{1} = 0 \cdot \mathbf{1}$. Since $\mathbf{1} \neq 0$, 0 is an eigenvalue with eigenvector $\mathbf{1}$.

**Q2: What is the physical / signal processing interpretation of the Dirichlet energy $x^T L x$?**
> **Answer:** $x^T L x = \frac{1}{2}\sum A_{ij}(x_i - x_j)^2$ measures the "smoothness" or total variation of signal $x$ across connected edges. Low energy means neighboring nodes have similar values (homophily), while high energy indicates high-frequency oscillations across edges.